# 교대자가 다시 열 수 있는 상담 원장

같은 문의가 닫혔다가 다시 접수됐다. 다음 근무자에게 넘길 목록을 원본 사건과 대조해 복원한다.

### 노트북 실행 준비

Python 3.12 커널에서 위쪽 셀부터 실행하세요. 이 예제는 파이썬 표준 라이브러리로 실행할 수 있습니다.

## 사건 한 줄과 문의 한 건

문의 ID는 한 고객 요청의 수명을 가리킨다. 사건 ID는 그 요청에 일어난 변경 한 번을 가리킨다. 문의 ID로 중복을 지우면 정당한 재접수까지 사라진다.

T1의 접수·완료·재접수가 세 줄이면 이력은 세 건, 현재 열린 문의는 한 건이다. 반면 같은 완료 사건을 통신 재시도로 두 번 받으면 이력에도 한 건만 남겨야 한다.

원본 사건을 순서대로 보존하고, 현재 상태를 별도 딕셔너리에서 갱신한다. 이 순서를 택하면 교대자가 “왜 다시 열렸나요?”라고 물을 때 사건까지 거슬러 갈 수 있다.

## 재전송 뒤에 재접수가 도착하면?

E2가 다시 들어와도 원장을 두 번 수정하지 않도록 사건 ID를 기억한다.

In [1]:
events = [("E1", "T1", "접수"), ("E2", "T1", "완료"), ("E2", "T1", "완료"), ("E3", "T1", "재접수")]
def rebuild(events):
    seen, state, history = set(), {}, []
    for event, ticket, action in events:
        if event in seen:
            continue
        seen.add(event)
        state[ticket] = action
        history.append(event)
    return state, history

print(rebuild(events))

({'T1': '재접수'}, ['E1', 'E2', 'E3'])


T1의 마지막 값은 재접수다. E2를 한 번만 보관했지만 E1과 E3까지 지우지는 않았다. 이 예제는 사건이 시간순으로 주어진다는 조건에서 실행한다.

### 도착 순서가 뒤바뀐 원장 복구

서버에서 내보낸 행에는 증가하는 순번이 있다. 재접수 순번 3이 완료 순번 2보다 먼저 도착했다. 현재 상태와 보존 이력을 복원한다.

In [2]:
rows = [(1, events[0]), (3, events[3]), (2, events[1])]

In [3]:
# 이 셀에 풀이를 작성하세요.

<details class="notebook-answer"><summary>풀이와 실행 결과</summary><pre><code>ordered = [record for number, record in sorted(rows)]
print(rebuild(ordered))</code></pre><pre>({&#x27;T1&#x27;: &#x27;재접수&#x27;}, [&#x27;E1&#x27;, &#x27;E2&#x27;, &#x27;E3&#x27;])
</pre><p>입력 순서에 의존하면 마지막 값이 완료가 된다. 이 파일에서는 서버가 붙인 순번을 기준으로 삼을 수 있다.</p></details>

## 상태를 결정할 수 없는 사건은 보류하기

같은 사건 번호로 서로 다른 처리 내역이 들어왔다. 먼저 도착한 행을 믿으면 입력 순서에 따라 결과가 바뀐다. 같은 번호의 내용이 전부 일치하는지 확인한 뒤 적용한다.

In [4]:
from collections import defaultdict
batch = [("E4","T2","접수"), ("E5","T2","완료"), ("E5","T2","재접수"), ("E6","T3","접수")]
variants = defaultdict(set)
for event, ticket, action in batch:
    variants[event].add((ticket, action))
conflicts = {event for event, choices in variants.items() if len(choices) > 1}
accepted = [row for row in batch if row[0] not in conflicts]
print(sorted(conflicts), rebuild(accepted)[0])

['E5'] {'T2': '접수', 'T3': '접수'}


E5 전체를 보류했기 때문에 T2에는 그 직전의 접수 상태가 남는다. 원본 두 줄을 버리지 말고 같은 사건 번호 아래에 묶어야 다음 근무자가 어느 시스템의 기록이 맞는지 확인할 수 있다.

### 교대 보고서에 원본 행 번호 남기기

CSV 머리글이 1행이다. 충돌 사건마다 재검토할 원본 행 번호와 사건 내용을 묶고, 동일한 내용이 한 번 더 전송돼도 충돌 종류가 늘지 않는지 검사한다.

In [5]:
review_rows = {}

In [6]:
# 이 셀에 풀이를 작성하세요.

<details class="notebook-answer"><summary>풀이와 실행 결과</summary><pre><code>review_rows = defaultdict(list)
for line, (event, ticket, action) in enumerate(batch, 2):
    if event in conflicts:
        review_rows[event].append((line,ticket,action))
print(review_rows[&quot;E5&quot;])</code></pre><pre>[(3, &#x27;T2&#x27;, &#x27;완료&#x27;), (4, &#x27;T2&#x27;, &#x27;재접수&#x27;)]
</pre><p>담당자는 3행과 4행을 바로 열 수 있다. 현재 상태 요약 한 줄로는 이 충돌을 복원할 수 없다.</p></details>

## 다른 입력에 적용하기

수업용 26행 원장에서 같은 사건 ID인데 시간이 다른 행을 별도로 모아 보자. 어느 값이 맞는지 결정할 자료가 없다면 현재 합계에 넣지 않고 원본 행 번호와 함께 교대자에게 넘긴다.

새 커널에서 CSV만 열어 정상·검토 행 수를 복원한다. 구현을 고쳤을 때 0분 완료가 다시 대기열에 들어가지 않는지도 시험한다.